# Challenge 5: Deploying Agents to Agent Platform

# **1 | Install Dependencies**

In [ ]:
!pip install --upgrade --quiet \
    "google-cloud-aiplatform[agent_engines,adk]>=1.112" \
    "google-adk[extensions]" \
    vertexai requests google-cloud-storage

# **2 | Imports and Configuration**

In [ ]:
import os
import asyncio
import logging
from typing import Optional

from google.adk.agents import Agent, SequentialAgent
from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmRequest, LlmResponse
from google.adk.tools.google_search_tool import google_search
from google.genai import types

# Suppress asyncio noise in output
logging.getLogger('asyncio').setLevel(logging.CRITICAL)

PROJECT_ID = "qwiklabs-gcp-01-ab542815eb6c"
LOCATION = "us-central1"
STAGING_BUCKET_NAME = f"{PROJECT_ID}-adk-staging"

# Use Vertex AI credentials instead of a standalone Gemini API key
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

MODEL_GEMINI = "gemini-2.5-flash"

print("Configuration complete.")
print(f"Staging bucket: {STAGING_BUCKET_NAME}")

# **3 | Initialize Vertex AI + Create Staging Bucket**

In [ ]:
import vertexai
from google.cloud import storage

# Create staging bucket if it doesn't exist
storage_client = storage.Client(project=PROJECT_ID)
try:
    storage_client.get_bucket(STAGING_BUCKET_NAME)
    print(f"Bucket gs://{STAGING_BUCKET_NAME} already exists.")
except Exception:
    print(f"Creating bucket gs://{STAGING_BUCKET_NAME}...")
    storage_client.create_bucket(STAGING_BUCKET_NAME, location=LOCATION)
    print("Bucket created.")

vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=f"gs://{STAGING_BUCKET_NAME}",
)
print(f"Vertex AI initialized with staging bucket: gs://{STAGING_BUCKET_NAME}")

# **4 | Define Callbacks**

In [ ]:
def log_before(
    callback_context: CallbackContext,
    llm_request: LlmRequest,
) -> Optional[LlmResponse]:
    """Log a preview of the input before it is sent to the model."""
    user_text = ""
    if llm_request.contents:
        last = llm_request.contents[-1]
        if last.parts:
            user_text = last.parts[0].text or ""
    print(f"[{callback_context.agent_name} \u2192 BEFORE MODEL] Input: {user_text[:120]!r}")
    return None


def log_after(
    callback_context: CallbackContext,
    llm_response: LlmResponse,
) -> Optional[LlmResponse]:
    """Log tool calls or a response preview after the model responds."""
    tool_calls = []
    response_text = ""
    if llm_response.content and llm_response.content.parts:
        for part in llm_response.content.parts:
            if hasattr(part, "text") and part.text:
                response_text = part.text
            if hasattr(part, "function_call") and part.function_call:
                tool_calls.append(part.function_call.name)
    if tool_calls:
        print(f"[{callback_context.agent_name} \u2192 AFTER MODEL] Tool call(s): {tool_calls}")
        return None
    if not response_text:
        return None
    preview = response_text[:200] + "..." if len(response_text) > 200 else response_text
    print(f"[{callback_context.agent_name} \u2192 AFTER MODEL] Response: {preview!r}")
    return None


print("Callbacks defined.")

# **5 | Agent Instructions**

In [ ]:
SEARCH_AGENT_INSTRUCTIONS = """
You are a research agent. Your job is to find accurate, up-to-date information
to answer the user's question.

Today's date is September 25, 2026.

Steps:
1. Use the google_search tool to find relevant information
2. Summarize the key facts clearly and concisely
3. Treat anything before September 25, 2026 as already having occurred
4. Always cite what you found — do not make up information

Return a clear, factual answer that other agents can build on.
"""

CRITIQUE_AGENT_INSTRUCTIONS = """
You are a critical review agent. You will receive a draft answer to a question.

Today's date is September 25, 2026. Events before this date have already occurred.
Do not flag past events as "future" or "upcoming".

Your job:
1. Identify any factual gaps, missing context, or unclear explanations
2. Note anything that could be more accurate, complete, or better structured
3. Suggest specific improvements — be constructive and precise

Do NOT rewrite the answer. Only provide a critique as a numbered list.
"""

REFINE_AGENT_INSTRUCTIONS = """
You are a refinement agent. You will receive:
- An original answer to a question
- A critique with suggestions for improvement

Your job:
1. Rewrite the answer incorporating all valid suggestions from the critique
2. Make it clear, accurate, well-structured, and complete
3. Do not introduce new information beyond what the search found

Return only the final polished answer — no preamble, no meta-commentary.
"""

ROOT_AGENT_INSTRUCTIONS = """
You are Erwin, a helpful assistant that answers questions thoroughly and accurately.

For every question you receive:
1. Pass it to the Erwin_AnswerTeam to research, critique, and refine the response
2. Return the final refined answer to the user

Always delegate to Erwin_AnswerTeam — do not answer directly yourself.
"""

print("Instructions defined.")

# **6 | Build All Agents**

In [ ]:
search_agent = Agent(
    name="Erwin_Search",
    model=MODEL_GEMINI,
    description="Searches the web to find accurate answers to questions.",
    instruction=SEARCH_AGENT_INSTRUCTIONS,
    tools=[google_search],
    before_model_callback=log_before,
    after_model_callback=log_after,
)

critique_agent = Agent(
    name="Erwin_Critique",
    model=MODEL_GEMINI,
    description="Reviews a draft answer and provides suggestions for improvement.",
    instruction=CRITIQUE_AGENT_INSTRUCTIONS,
    before_model_callback=log_before,
    after_model_callback=log_after,
)

refine_agent = Agent(
    name="Erwin_Refine",
    model=MODEL_GEMINI,
    description="Rewrites and polishes an answer based on critique feedback.",
    instruction=REFINE_AGENT_INSTRUCTIONS,
    before_model_callback=log_before,
    after_model_callback=log_after,
)

answer_team = SequentialAgent(
    name="Erwin_AnswerTeam",
    description="A sequential workflow that searches, critiques, and refines answers.",
    sub_agents=[search_agent, critique_agent, refine_agent],
)

root_agent = Agent(
    name="Erwin_Root",
    model=MODEL_GEMINI,
    description="Receives user questions and delegates to the Erwin_AnswerTeam workflow.",
    instruction=ROOT_AGENT_INSTRUCTIONS,
    sub_agents=[answer_team],
    before_model_callback=log_before,
    after_model_callback=log_after,
)

print("All agents created.")

# **7 | Wrap in AdkApp + Local Test**

In [ ]:
from vertexai.preview.reasoning_engines import AdkApp
from IPython.display import Markdown, display

app = AdkApp(agent=root_agent)

print("AdkApp created. Running local test...")
print("-" * 40)
print("Query: What is the Google Agent Development Kit and what are its main features?\n")

current_author = None
final_text = ""

for event in app.stream_query(
    user_id="local-test-user",
    message="What is the Google Agent Development Kit and what are its main features?",
):
    if isinstance(event, dict):
        author = event.get("author", "")
        content = event.get("content", {})
        parts = content.get("parts", []) if isinstance(content, dict) else []
    else:
        author = getattr(event, "author", "")
        content = getattr(event, "content", None)
        parts = getattr(content, "parts", []) if content else []

    for part in parts:
        if isinstance(part, dict):
            text = part.get("text")
        else:
            text = getattr(part, "text", None)

        if text:
            if author != current_author:
                display_name = author.replace("_", " ").title() if author else "System"
                print(f"\n[{display_name}]")
                current_author = author
            print(text)
            final_text = text

print("\n" + "=" * 40)
print("LOCAL TEST COMPLETE")
print("=" * 40)

# **8 | Deploy to Agent Platform**

In [ ]:
from vertexai import agent_engines

print("Deploying to Agent Platform... (this takes 2-5 minutes)")
print("-" * 40)

remote_agent = agent_engines.create(
    app,
    requirements=[
        "google-cloud-aiplatform[agent_engines,adk]>=1.112",
        "google-adk[extensions]==2.10.0",
    ],
    display_name="Erwin Answer Workflow Agent",
    description="Search, critique, and refine workflow agent built with Google ADK.",
)

print(f"\nDeployment complete!")
print(f"Resource name: {remote_agent.resource_name}")

# **9 | Test: Remote Deployed Agent**

In [ ]:
import time
from IPython.display import Markdown, display

RETRY_DELAYS = [15, 30, 60]

test_queries = [
    "What is the Google Agent Development Kit and what are its main features?",
    "What caused the 2008 financial crisis?",
]

print("=" * 60)
print("TEST: DEPLOYED AGENT ON AGENT PLATFORM")
print("=" * 60)

for query in test_queries:
    print(f"\nQuery: {query}")
    print("-" * 40)

    current_author = None

    for attempt, delay in enumerate([0] + RETRY_DELAYS):
        if delay > 0:
            print(f"  [Error — retrying in {delay}s (attempt {attempt}/{len(RETRY_DELAYS)})...]")
            time.sleep(delay)
        try:
            for event in remote_agent.stream_query(
                user_id="remote-test-user",
                message=query,
            ):
                # Remote agent events can be dicts or objects — handle both
                if isinstance(event, dict):
                    author = event.get("author", "")
                    content = event.get("content", {})
                    parts = content.get("parts", []) if isinstance(content, dict) else []
                else:
                    author = getattr(event, "author", "")
                    content = getattr(event, "content", None)
                    parts = getattr(content, "parts", []) if content else []

                for part in parts:
                    if isinstance(part, dict):
                        text = part.get("text")
                    else:
                        text = getattr(part, "text", None)

                    if text:
                        if author != current_author:
                            display_name = author.replace("_", " ").title() if author else "System"
                            print(f"\n[{display_name}]")
                            current_author = author
                        print(text)
            break  # success — exit retry loop
        except Exception as e:
            if attempt < len(RETRY_DELAYS):
                continue  # retry on any error
            else:
                print(f"  [Failed after all retries: {str(e)[:200]}]")
                break

    print()

# **10 | Cleanup (Optional)**

In [ ]:
# Uncomment to delete the remote agent and free up resources
# print(f"Deleting remote agent: {remote_agent.resource_name}")
# remote_agent.delete(force=True)
# print("Remote agent deleted. Cleanup complete.")

print(f"Remote agent still running: {remote_agent.resource_name}")
print("Uncomment the lines above to delete it when you are done.")